# Pump-probe fluence & time resolution (grazing incidence)

Two small calculators for planning grazing-incidence pump-probe experiments,
ported from [`calx.pumpprobe`](https://github.com/htlemke/calx) (pure
numpy/scipy, no facility-specific dependencies, so reimplemented directly in
`bernina_prep.timing` rather than depending on the full `calx` package).

## Absorbed fluence

Pulse energy spread over the beam's footprint on the sample surface. At
grazing incidence (`angle` well below 90&deg;) the same beam energy is
spread over a much larger illuminated area than the beam's own
cross-section, reducing the fluence per unit area accordingly.

In [ ]:
import sys
from pathlib import Path

try:
    import bernina_prep  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import ipywidgets as widgets
from IPython.display import display

from bernina_prep.timing import fluence, grazing_incidence_time_resolution


def fluence_ui(energy_uj=100.0, area_mm2=0.01, angle_deg=90.0, reflectivity=0.0):
    f = fluence(
        energy_uj * 1e-6,
        area_mm2 * 1e-6,
        np.deg2rad(angle_deg),
        reflectivity,
    )
    print(f"Absorbed fluence: {f:.4f} mJ/cm\u00b2")


fluence_ctrl = {
    "energy_uj": widgets.FloatText(value=100.0, description="pulse energy [\u00b5J]"),
    "area_mm2": widgets.FloatText(value=0.01, step=0.001, description="beam area [mm\u00b2]"),
    "angle_deg": widgets.FloatSlider(value=90.0, min=0.1, max=90.0, step=0.1, description="incidence angle [deg]"),
    "reflectivity": widgets.FloatSlider(value=0.0, min=0.0, max=0.99, step=0.01, description="reflectivity"),
}
fluence_out = widgets.interactive_output(fluence_ui, fluence_ctrl)
display(widgets.VBox(list(fluence_ctrl.values()) + [fluence_out]))


## Geometric time resolution at grazing incidence

Independent of the pump/probe pulse durations themselves, a **purely
geometric** timing smear arises whenever pump and probe cross the sample
surface at an angle to each other with a finite footprint: different points
across that footprint see the pump wavefront arrive at slightly different
times, because of the path-length difference across the (often strongly
elongated, at grazing incidence) illuminated area.

The sign of the result indicates *which* beam's wavefront leads across the
footprint (pump vs. probe); the **magnitude** is the relevant smearing
contribution to add (in quadrature with the pulse durations and other
jitter sources) to get the overall time resolution.

In [ ]:
def timeres_ui(probe_size_mm=0.5, probe_angle_deg=0.5, pump_angle_deg=5.0):
    t = grazing_incidence_time_resolution(
        probe_size_mm * 1e-3,
        np.deg2rad(probe_angle_deg),
        np.deg2rad(pump_angle_deg),
    )
    print(f"Geometric time smear: {t * 1e15:+.2f} fs  (magnitude: {abs(t) * 1e15:.2f} fs)")


timeres_ctrl = {
    "probe_size_mm": widgets.FloatText(value=0.5, step=0.01, description="probe size [mm]"),
    "probe_angle_deg": widgets.FloatText(value=0.5, step=0.05, description="probe angle [deg]"),
    "pump_angle_deg": widgets.FloatText(value=5.0, step=0.1, description="pump angle [deg]"),
}
timeres_out = widgets.interactive_output(timeres_ui, timeres_ctrl)
display(widgets.VBox(list(timeres_ctrl.values()) + [timeres_out]))
